## Import stuff

In [ ]:
import pandas as pd

#stock data
stock_data = pd.read_csv('data/Tesla_stock_data_filled.csv', parse_dates=['Date'])

#headlines data
# headlines_data = pd.read_csv('data/tesla_headlines.csv', parse_dates=['pub_date'])

print(stock_data.head())
# print(headlines_data.head())

In [ ]:
import pandas as pd
import glob

# Match files like 1718tesla_headlines.csv, 1920tesla_headlines.csv, ..., 2324tesla_headlines.csv
file_list = glob.glob("data/[0-9][0-9][0-9][0-9]tesla_headlines.csv")  

# Read and combine all CSVs
df_list = [pd.read_csv(file, parse_dates=['pub_date']) for file in file_list]
headlines_data = pd.concat(df_list, ignore_index=True)

# Sort by date (important for time series analysis)
headlines_data = headlines_data.sort_values(by='pub_date').reset_index(drop=True)

# Check combined data
print(headlines_data.info())
print(headlines_data.head())

## TextBlob model (corr = 0.084)

In [ ]:
# from textblob import TextBlob

# # Add a sentiment score to the headlines
# def get_sentiment_score(text):
#     analysis = TextBlob(text)
#     return analysis.sentiment.polarity  # Returns a score between -1 (negative) and 1 (positive)

# headlines_data['sentiment'] = headlines_data['headline'].apply(get_sentiment_score)
# print(headlines_data[['headline', 'sentiment']].head(100))

## VADER model (corr = 0.032)

In [ ]:
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# # Initialize VADER
# analyzer = SentimentIntensityAnalyzer()

# # Sentiment function using VADER
# def get_sentiment_score(text):
#     return analyzer.polarity_scores(text)['compound']  # Compound score ranges from -1 to 1

# headlines_data['sentiment'] = headlines_data['headline'].apply(get_sentiment_score)
# print(headlines_data[['headline', 'sentiment']].head(100))

## Vader Lexicon

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Download resources if not already
nltk.download('punkt')
nltk.download('stopwords')

# Initialize VADER to access its lexicon
analyzer = SentimentIntensityAnalyzer()
vader_lexicon = analyzer.lexicon  # This is the VADER lexicon as a dict

# Optional: NLTK stopwords
stop_words = set(stopwords.words('english'))

# Custom sentiment scoring using raw VADER lexicon
def get_sentiment_score_from_lexicon(text):
    words = word_tokenize(text.lower())
    words = [word for word in words if word.isalpha() and word not in stop_words]
    
    score = 0
    for word in words:
        if word in vader_lexicon:
            score += vader_lexicon[word]
    
    # Normalize score (like VADER's compound, rough approximation)
    if len(words) > 0:
        normalized_score = score / len(words)
    else:
        normalized_score = 0.0
    
    return normalized_score

# Apply to your DataFrame
headlines_data['sentiment'] = headlines_data['headline'].apply(get_sentiment_score_from_lexicon)

# Show some results
print(headlines_data[['headline', 'sentiment']].head(100))


## FinBert Model (corr=0.060)

In [ ]:
# from transformers import BertTokenizer, BertForSequenceClassification
# import torch

# # Set device to MPS for Apple Silicon
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# print(f"Using device: {device}")

# # Load FinBERT model
# model_name = "ProsusAI/finbert"
# tokenizer = BertTokenizer.from_pretrained(model_name)
# model = BertForSequenceClassification.from_pretrained(model_name).to(device)

# # Function for FinBERT sentiment analysis
# def get_sentiment_score(text):
#     inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
#     with torch.no_grad():  # Disable gradient calculations for inference
#         outputs = model(**inputs)
#     probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
#     sentiment_score = probs[0][2] - probs[0][0]  # Positive - Negative Score
#     return sentiment_score.item()

# # Apply sentiment analysis to headlines
# headlines_data['sentiment'] = headlines_data['headline'].apply(get_sentiment_score)

# # Print results
# print(headlines_data[['headline', 'sentiment']].head(100))

## Plotting the sentiment scores

In [ ]:
import matplotlib.pyplot as plt

# Aggregate sentiment by day
headlines_data['date'] = pd.to_datetime(headlines_data['pub_date'])  # Ensure dates are in datetime format
daily_sentiment = headlines_data.groupby('date')['sentiment'].mean()

# Plot sentiment trend
plt.figure(figsize=(10, 6))
daily_sentiment.plot(title="Daily Sentiment Trend", xlabel="Date", ylabel="Sentiment Score")
plt.show()

## Changing stock data to percent change

In [ ]:
# Ensure data is sorted by date
stock_data = stock_data.sort_values(by='Date')

# Shift 'Close' to get the next day's closing price
stock_data['next_day_close'] = stock_data['Close'].shift(-1)

# Compute percent change from today's close to next day's close
stock_data['pct_change'] = ((stock_data['next_day_close'] - stock_data['Close']) / stock_data['Close']) * 100

stock_data = stock_data.dropna()
print(stock_data[['Date', 'Close', 'next_day_close', 'pct_change']].head(10))

## Plotting and finding correlation between sentiment scores and stock price

In [ ]:
# Convert both date columns to the same format
headlines_data['pub_date'] = pd.to_datetime(headlines_data['pub_date']).dt.date
stock_data['Date'] = pd.to_datetime(stock_data['Date']).dt.date

# Merge datasets on dates
merged_data = pd.merge(
    headlines_data,  # Dataset with sentiment scores
    stock_data,      # Dataset with stock prices
    left_on='pub_date',  # Match 'pub_date' in headlines_data
    right_on='Date',     # Match 'Date' in stock_data
    how='inner'          # Perform an inner join to keep only matching dates
)

# Calculate correlation for numeric columns
correlation = merged_data[['sentiment', 'pct_change']].corr()
print("Correlation between sentiment and stock close price percent change:")
print(correlation)

# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(merged_data['sentiment'], merged_data['pct_change'], alpha=0.6, color='blue')
plt.title("Sentiment vs Stock Close Price")
plt.xlabel("Sentiment Score")
plt.ylabel("Close Price")
plt.grid(True)
plt.show()

## Check for correlation with lags

In [ ]:
# test for lag in stock market
for lag in [1, 2, 7]:
    merged_data[f'Close_lag{lag}'] = merged_data['Close'].shift(lag)
    print(merged_data[['sentiment', f'Close_lag{lag}']].corr())

## VAR Model (Time Series Modeling)

In [ ]:
import pandas as pd
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt

# Ensure datetime index and sorted order
merged_data.index = pd.to_datetime(merged_data.index)
merged_data.sort_index(inplace=True)

# Drop any rows with missing values
merged_data.dropna(inplace=True)

# Check the structure
print(merged_data.head())

In [ ]:
# Step 1: Reset the bad index
merged_data.reset_index(drop=True, inplace=True)

# Step 2: Make sure the correct date column is datetime
merged_data['Date'] = pd.to_datetime(merged_data['Date'], errors='coerce')

# Step 3: Drop rows with invalid dates (if any)
merged_data.dropna(subset=['Date'], inplace=True)

# Step 4: Set 'Date' as the index
merged_data.set_index('Date', inplace=True)

# Step 5: Sort index for time series modeling
merged_data.sort_index(inplace=True)

In [ ]:
# print(merged_data.index)
# print(merged_data.head())

In [ ]:
from statsmodels.tsa.api import VAR

# Choose variables for VAR — e.g., sentiment and price-related metrics
var_df = merged_data[['sentiment', 'Close', 'pct_change']].dropna()

train = var_df[:-30]  # last 30 days as test
test = var_df[-30:]

In [ ]:
model = VAR(train)

# Check lag order
lag_order = model.select_order(maxlags=15)
print(lag_order.summary())

In [ ]:
best_lag = lag_order.aic  # you can also use bic or fpe
model_fitted = model.fit(best_lag)
print(model_fitted.summary())

In [ ]:
# Get forecasts for the same number of steps as the test set
forecast_input = train.values[-best_lag:]
forecast = model_fitted.forecast(y=forecast_input, steps=30)

# Convert forecast to DataFrame
forecast_df = pd.DataFrame(forecast, index=test.index, columns=var_df.columns)

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(test['Close'], label='Actual Close')
plt.plot(forecast_df['Close'], label='Forecasted Close')
plt.legend()
plt.title('VAR Model - Tesla Close Price Prediction')
plt.show()